In [1]:
# Cell 01 | Open the persistent SQLite search index
# Purpose: Connect to the existing SQLite-backed FAQ index without downloading or rebuilding the knowledge base.
# Key points: Persistent storage 讓新的 Python session 可以直接重用已完成的 ingestion 結果；query process 不需要重新執行 data loading 或 indexing。
# Execution: 需先完成 02_persistent_rag_ingest.ipynb；執行後應直接從既有 faq.db 讀到 139 documents。

from sqlitesearch import TextSearchIndex


sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db",
)

print("Documents available:", sqlite_index.count())

Documents available: 139


In [2]:
# Cell 02 | Search the persistent FAQ index
# Purpose: Retrieve relevant FAQ documents directly from the persistent SQLite search index.
# Key points: Query process 直接搜尋既有 faq.db，不需要重新下載或建立 index；搜尋介面與先前 MinSearch 的使用方式相近。
# Execution: 需先完成 Cell 01；執行後應回傳與「課程開始後是否還能加入」高度相關的 FAQ documents。

query = "Can I still join the course after it started?"

search_results = sqlite_index.search(
    query=query,
    boost_dict={
        "question": 2.0,
        "section": 0.5,
    },
    filter_dict={
        "course": "llm-zoomcamp",
    },
    num_results=5,
)

print("Results returned:", len(search_results))

for rank, doc in enumerate(search_results, start=1):
    print(f"\nRank {rank}")
    print("Section:", doc["section"])
    print("Question:", doc["question"])

Results returned: 5

Rank 1
Section: General Course-Related Questions
Question: I just discovered the course. Can I still join?

Rank 2
Section: General Course-Related Questions
Question: The homework submission form is still open even though the deadline has passed — can I still submit?

Rank 3
Section: General Course-Related Questions
Question: I missed the first homework - can I still get a certificate?

Rank 4
Section: General Course-Related Questions
Question: Certificate: Can I follow the course in a self-paced mode and get a certificate?

Rank 5
Section: General Course-Related Questions
Question: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?


In [3]:
# Cell 03 | Connect the persistent index to RAGBase
# Purpose: Reuse the existing RAG pipeline with the persistent SQLite search index.
# Key points: RAGBase 不需要知道 index 是 MinSearch 還是 SQLiteSearch；只要 backend 提供相容的 search() interface，就能直接替換。
# Execution: 需先完成 Cell 01；執行後應成功建立 assistant，不會重新下載 FAQ，也不會重新 indexing。

from dotenv import load_dotenv
from openai import OpenAI

from rag_helper import RAGBase


load_dotenv("../.env")

client = OpenAI()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=client,
)

print("Persistent RAG assistant ready.")

Persistent RAG assistant ready.


In [4]:
# Cell 04 | Run RAG with the persistent index
# Purpose: Run the complete RAG pipeline using FAQ documents retrieved from the persistent SQLite index.
# Key points: Query → SQLite retrieval → context → prompt → LLM；RAGBase 邏輯不需因 search backend 更換而重寫。
# Execution: 需先完成 Cell 01–03；執行後應根據 FAQ context 回答課程開始後是否仍能加入。

query = "Can I still join the course after it started?"

answer = assistant.rag(query)

print("Question:")
print(query)

print("\nAnswer:")
print(answer)

Question:
Can I still join the course after it started?

Answer:
Yes, you can still join after the course has started. To receive a certificate, you must submit and pass the capstone project and complete the required peer reviews while the live cohort is still accepting submissions.


In [6]:
# Cell 05 | Trace the retrieved evidence
# Purpose: Inspect the FAQ documents used to ground the persistent RAG answer.
# Key points: RAG 的答案品質取決於 retrieval evidence；本格將 answer 與 Top-K documents 分開檢查，確認主要敘述是否有來源支持。
# Execution: 使用與 Cell 04 相同的 query；執行後應列出 Top-5 retrieved documents 的 section、question 與 answer。

search_results = assistant.search(query)

for rank, doc in enumerate(search_results, start=1):
    print(f"\nRank {rank}")
    print("Section:", doc["section"])
    print("Question:", doc["question"])
    print("Answer:", doc["answer"])


Rank 1
Section: General Course-Related Questions
Question: I just discovered the course. Can I still join?
Answer: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

Rank 2
Section: General Course-Related Questions
Question: The homework submission form is still open even though the deadline has passed — can I still submit?
Answer: Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit.

Rank 3
Section: General Course-Related Questions
Question: I missed the first homework - can I still get a certificate?
Answer: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

Rank 4
Section:

In [8]:
# Cell 06 | Verify the unknown-answer boundary
# Purpose: Confirm that the persistent RAG pipeline refuses to invent an answer when the retrieved FAQ context does not contain the requested information.
# Key points: Grounded RAG 必須同時具備「有證據時回答」與「沒有證據時拒絕臆測」兩種行為；這也是 backend 更換後的重要 regression test。
# Execution: 使用與課程 FAQ 無關的問題；預期回答應明確表示 context 中沒有足夠資訊，而不是提供真實世界答案。

unknown_query = "What is the current weather in Taipei?"

unknown_answer = assistant.rag(unknown_query)

print("Question:")
print(unknown_query)

print("\nAnswer:")
print(unknown_answer)

Question:
What is the current weather in Taipei?

Answer:
I don’t know. The provided context explains how to configure a weather tool, but it does not include current weather data for Taipei.


In [10]:
# Cell 07 | Close the persistent search index
# Purpose: Close the SQLite connection after completing retrieval and RAG verification.
# Key points: Persistent data remains stored in faq.db after the connection is closed；connection lifecycle 與資料 persistence 是兩件不同的事。
# Execution: 在所有 query 與 RAG tests 完成後執行；應正常關閉 SQLite connection，不刪除已建立的 database。

sqlite_index.close()

print("Persistent search index closed.")

Persistent search index closed.
